## Experimento 3:
1. Aumento do Batch de 32 para 64
2. Obs.: loss mais estável e menos update por época

In [1]:
# Importação das bibliotecas e inicialização dos dados
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, classification_report
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device) # Onde o programa vai ser executado, se existir uma gpu compatível com cuda (nvidia) disponível, usa a GPU (mais rapido), se não vai para cpu

train_path = "dataset_processed/train"
test_path  = "dataset_processed/test"

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

cpu


In [2]:
# Dados de Treino
train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
test_dataset  = datasets.ImageFolder(test_path, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True) # MUDANÇA
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False) # MUDANÇA

classes = train_dataset.classes
print(classes)

['asphalt', 'belgian_blocks', 'offroad']


In [3]:
# Função Treino
def train_model(model, epochs=10):

    model.to(device)

    criterion = nn.CrossEntropyLoss() # mede quão longe a previsão do modelo está da resposta correta (padrão para multiclasse)
    optimizer = optim.Adam(model.parameters(), lr=0.001) # atualiza os pesos da rede para reduzir o erro

    start = time.time()

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad() # zera gradientes antigos
            outputs = model(images) # as imagens entram no modelo
            loss = criterion(outputs, labels) # compara as saidas com o gabarito e calcula a loss
            loss.backward() # calcuça quanto cada peso contribuiu para o erro
            optimizer.step() # atualiza os pesos

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss:.4f}")

    end = time.time()

    print("Tempo treino:", round(end-start,2), "segundos")

In [4]:
# Função Teste
def evaluate_model(model):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images) # gera as previsões
            _, preds = torch.max(outputs, 1) # escolhe a classe com o maior valor

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred) # calcula a porcentaggem de acertos

    print("Accuracy:", accuracy)
    print(classification_report(y_true, y_pred, target_names=classes)) # mostra as métricas por classe

In [5]:
# Avaliação do ResNet18
print("\n##### RESNET18 #####")

resnet = models.resnet18(weights="DEFAULT") # carrega o modelo
resnet.fc = nn.Linear(resnet.fc.in_features, 3) # muda para 3 classes

train_model(resnet, epochs=10) # treina
evaluate_model(resnet) # testa

# Avaliação do EfficientNet-B0
print("\n##### EFFICIENTNET_B0 #####")

effnet = models.efficientnet_b0(weights="DEFAULT")
effnet.classifier[1] = nn.Linear(
    effnet.classifier[1].in_features, 3
)

train_model(effnet, epochs=10)
evaluate_model(effnet)


##### RESNET18 #####
Epoch 1/10 Loss: 4.0722
Epoch 2/10 Loss: 1.9429
Epoch 3/10 Loss: 4.0868
Epoch 4/10 Loss: 8.8814
Epoch 5/10 Loss: 2.8948
Epoch 6/10 Loss: 3.7743
Epoch 7/10 Loss: 2.5841
Epoch 8/10 Loss: 3.4705
Epoch 9/10 Loss: 3.6075
Epoch 10/10 Loss: 2.3859
Tempo treino: 467.8 segundos
Accuracy: 0.85
                precision    recall  f1-score   support

       asphalt       0.84      0.99      0.91       218
belgian_blocks       0.82      0.28      0.42        32
       offroad       0.91      0.60      0.72        50

      accuracy                           0.85       300
     macro avg       0.86      0.62      0.68       300
  weighted avg       0.85      0.85      0.83       300


##### EFFICIENTNET_B0 #####
Epoch 1/10 Loss: 4.2346
Epoch 2/10 Loss: 1.6506
Epoch 3/10 Loss: 0.6527
Epoch 4/10 Loss: 0.7154
Epoch 5/10 Loss: 0.3398
Epoch 6/10 Loss: 0.5488
Epoch 7/10 Loss: 1.1932
Epoch 8/10 Loss: 0.7859
Epoch 9/10 Loss: 0.3754
Epoch 10/10 Loss: 0.8601
Tempo treino: 474.69 segundo

## Analise dos Resultados:
### Experimento 3 – Aumento do Batch Size

Mudança:
```
32 → 64
```
Objetivo:
Verificar se batches maiores melhorariam estabilidade e eficiência

### ResNet18 + Batch 64:

| Métrica  | Melhor Config (Exp2) | Exp3 | Variação |
| -------- | -------------------- | ---- | -------- |
| Accuracy | 0.93                 | 0.85 | -0.08    |
| Macro F1 | 0.85                 | 0.68 | -0.17    |

-> Queda significativa de desempenho. O modelo perdeu capacidade de generalização.


### EfficientNet-B0 + Batch 64

| Métrica  | Melhor Config (Exp2) | Exp3 | Variação |
| -------- | -------------------- | ---- | -------- |
| Accuracy | 0.93                 | 0.89 | -0.04    |
| Macro F1 | 0.84                 | 0.76 | -0.08    |

-> Também houve piora, mas, foi menor que no ResNet18.